# EgoLanes

## I. Getting Started Guide

### 1. EgoLanes Model Introduction

EgoLanes is a specialized segmentation network within the VisionPilot suite designed to detect and segment ego-lane
boundaries without relying on HD maps. It follows an Encoder-Context-Decoder pattern, optimized for spatial precision
and global road awareness.

The architecture components include:

- A pretrained EfficientNet-B0 Backbone (Encoder) to extract hierarchical features at five different scales, capturing
everything from fine textures to broad semantic shapes.

- A multi-scale feature fusion layer that downsamples early high-resolution features to a uniform spatial size
($10 \times 20$ in the default configuration), concatenating them into a "deep feature" block.

- Context MLP layers to compress global information and then projects it back into the feature maps using a residual
attention mechanism ($Context \times Features + Features$).

- EgoPath Neck (Decoder) which upsamples the contextualized features back to the target resolution. It uses Skip
Connections from the encoder to re-inject fine-grained spatial data that was lost during downsampling.

- Segmentation Head consisting of 3 convolutional layers produce a 3-channel output, typically representing lane classes
or boundary masks.

As a lane boundary detection model, EgoLanes processes raw image frames and performs real-time semantic segmentation of
driving lanes in the image. It produces a three class segmentation output for the ego-left lane, the ego-right lane and
all other lanes. It outputs lanes at 1/4 resolution of the input image size allowing for quick inference on low power
embedded hardware. EgoLanes was trained with data from a variety of real-world datasets including TuSimple, OpenLane,
CurveLanes, Jiqing, and ONCE3D Lane.

![](../../Media/EgoLanes_GIF_2.gif)

### 2. Environment Setup

**If you have done these steps of environment setup in other End-to-End Model Tutorials, please skip this and move forward to other sections.**

First, we need to clone the repository to your local environment. We will use the official repository from the
Autoware Foundation.

In [ ]:
import os

# Clone Autoware VisionPilot repo, if not yet done
if not os.path.exists("./autoware_vision_pilot"):
    !git clone https://github.com/autowarefoundation/autoware_vision_pilot.git

Cloning into 'autoware_vision_pilot'...
remote: Enumerating objects: 20876, done.
remote: Counting objects: 100% (5314/5314), done.
remote: Compressing objects: 100% (2352/2352), done.
remote: Total 20876 (delta 3015), reused 3031 (delta 2957), pack-reused 15562 (from 2)
Receiving objects: 100% (20876/20876), 290.06 MiB | 16.23 MiB/s, done.
Resolving deltas: 100% (11946/11946), done.
/home/tranhuunhathuy/Documents/Autoware/autoware.privately-owned-vehicles/Tutorials/E2E_Models/autoware_vision_pilot


/home/tranhuunhathuy/Documents/Autoware/autoware.privately-owned-vehicles/test/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


The VisionPilot project relies on several key libraries, including **PyTorch** for model execution and
**OpenCV** for image processing.

We will use `pip` to install the required dependencies directly from the `requirements.txt` file provided in 
the `Models` directory.

In [ ]:
# Install required dependencies
%pip install --upgrade pip
%pip install -r autoware_vision_pilot/Models/requirements.txt

# Verify the installation (checking torch as a primary dependency)
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA availability: {torch.cuda.is_available()}")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
PyTorch version: 2.7.0+cu126
CUDA availability: True


### 3. Download Models

You can manually download the model weight files, using the links below. Subsequent tutorial sections assume
you already downloaded them and put them inside directory: 
`autoware_vision_pilot/Tutorials/E2E_Models/weights/`.

- [Link to Download Pytorch Model Weights (*.pth)](https://drive.google.com/file/d/1Njo9EEc2tdU1ffo8AUQ9mjwuQ9CzSRPX/view?usp=sharing)
- [Link to Download ONNX FP32 Moel Height (*.onnx)](https://drive.google.com/file/d/1b4jAoH6363ggTgVU0b6URbFfcOL3-r1Q/view?usp=sharing)

You can also use below script block, which uses `gdown` to download above model weights into such default directory.

In [ ]:
%pip install gdown

import gdown
import os

model_dir = "./weights"
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

metadata = {
    "pth"   : {
        "url"       : "https://drive.google.com/uc?id=1Njo9EEc2tdU1ffo8AUQ9mjwuQ9CzSRPX",
        "filepath"  : os.path.join(os.getcwd(), model_dir, "egolanes.pth")
    },
    "onnx"  : {
        "url"       : "https://drive.google.com/uc?id=1b4jAoH6363ggTgVU0b6URbFfcOL3-r1Q",
        "filepath"  : os.path.join(os.getcwd(), model_dir, "egolanes.onnx")
    }
}

for weight_type in metadata:

    url         = metadata[weight_type]["url"]
    filepath    = metadata[weight_type]["filepath"]

    if not os.path.exists(filepath):
        print(f"Downloading EgoLanes {weight_type} weights...")
        gdown.download(url, filepath, quiet = False)
    else:
        print(f"EgoLanes {weight_type} weights already exist at {filepath}. Skipping download.")

Note: you may need to restart the kernel to use updated packages.


Downloading...
From (original): https://drive.google.com/uc?id=1Njo9EEc2tdU1ffo8AUQ9mjwuQ9CzSRPX
From (redirected): https://drive.google.com/uc?id=1Njo9EEc2tdU1ffo8AUQ9mjwuQ9CzSRPX&confirm=t&uuid=5f0df7d5-491e-4b5a-ab64-4795c4264d50
To: /home/tranhuunhathuy/Documents/Autoware/autoware.privately-owned-vehicles/Tutorials/E2E_Models/autoware_vision_pilot/Tutorials/E2E_Models/weights/egolanes.pth
100%|██████████| 208M/208M [00:12<00:00, 16.6MB/s] 


Downloading...
From (original): https://drive.google.com/uc?id=1b4jAoH6363ggTgVU0b6URbFfcOL3-r1Q
From (redirected): https://drive.google.com/uc?id=1b4jAoH6363ggTgVU0b6URbFfcOL3-r1Q&confirm=t&uuid=12fe3426-1fd7-429c-bc3f-da2fc3e11109
To: /home/tranhuunhathuy/Documents/Autoware/autoware.privately-owned-vehicles/Tutorials/E2E_Models/autoware_vision_pilot/Tutorials/E2E_Models/weights/egolanes.onnx
100%|██████████| 208M/208M [00:14<00:00, 14.1MB/s] 


## II. Quick Test/Inference

Pre-requisites: please ensure that you have completed the above **Environment Setup** first.

### 1. Image Inference

### 2. Video Inference

## III. Model Training

### 1. Model Training on New Data

### 2. Building an Inference Pipeline